# 03 - Phase 2: retrieval layer and the nearest-neighbour baseline (B0)

> Run with **Save Version -> Save & Run All (Commit)** (interactive output is lost when the session ends).
> Settings: **Accelerator = GPU T4** (one is enough), **Internet = On** (downloads the embedding models from Hugging Face).

**Inputs (Add Input)**
1. the Phase 0 dataset: `train_processed.csv` and `test_processed.csv` (the latter is the unseen 1,000-question set)
2. the chunks dataset: `chunks.jsonl.gz` (from Phase 1)

**What it builds**
* **Index A** - training questions: BM25 + `bge-m3` dense, fused with reciprocal-rank fusion (RRF). Tuned on the dev split.
  Pool = the 12,825 *train* rows only (so dev/test measurements are leak-free); the unseen set uses all 15,000 rows.
* **Index B** - PDF chunks: BM25 + `bge-small-en-v1.5` dense, fused with RRF. Quality is measured against the pages the
  training answers actually cite (page-hit / doc-hit @ m).
* **B0 baseline** - answer a question with the answer of its nearest training question (and a medoid-of-top-5 variant),
  scored on the 1,500 hold-out rows with the shared evaluation harness.
* **Artifacts for Phase 4**: top-10 neighbour questions and top-20 chunks for every train/dev/test/unseen question, plus all embeddings.

In [1]:
!pip install -q rouge-score sacrebleu sentence-transformers

In [2]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|\u2013|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

Writing eval_harness.py


In [3]:
%%writefile bm25.py
"""Sparse BM25 (Okapi) that scales to hundreds of thousands of documents.

Term weights are precomputed into a sparse matrix, so scoring a batch of queries is one sparse matrix product.
Pure ASCII source on purpose.
"""
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer


class BM25:
    def __init__(self, k1=1.2, b=0.75, stop_words="english"):
        self.k1, self.b, self.stop_words = k1, b, stop_words

    def fit(self, texts):
        self.cv = CountVectorizer(lowercase=True, stop_words=self.stop_words, token_pattern=r"(?u)\b\w+\b",
                                  dtype=np.float32)
        X = self.cv.fit_transform(texts).tocsr()
        n = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
        idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
        avg = max(1e-9, float(dl.mean()))
        rows = np.repeat(np.arange(n), np.diff(X.indptr))
        tf = X.data
        X.data = (idf[X.indices] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * dl[rows] / avg))
                  ).astype(np.float32)
        self.W = X.T.tocsr()  # (vocab, n_docs)
        self.n = n
        return self

    def search(self, queries, k=50, batch=128):
        """Returns (idx, score), each (n_queries, k); idx is -1 where a query shares no term with the document."""
        Q = self.cv.transform(queries).tocsr()
        Q.data[:] = 1.0
        k = min(k, self.n)
        idx = np.full((Q.shape[0], k), -1, dtype=np.int32)
        sc = np.zeros((Q.shape[0], k), dtype=np.float32)
        for i in range(0, Q.shape[0], batch):
            S = (Q[i:i + batch] @ self.W).toarray()
            part = np.argpartition(-S, k - 1, axis=1)[:, :k]
            ps = np.take_along_axis(S, part, axis=1)
            order = np.argsort(-ps, axis=1)
            idx[i:i + batch] = np.take_along_axis(part, order, axis=1)
            sc[i:i + batch] = np.take_along_axis(ps, order, axis=1)
        idx[sc <= 0] = -1
        return idx, sc

Writing bm25.py


In [4]:
%%writefile encoders.py
"""Thin wrapper around sentence-transformers so the notebook (and tests) can swap the encoder easily.
Pure ASCII source on purpose."""
import numpy as np

_MODELS = {}


def encode(model_name, texts, batch_size=64, max_len=None, prefix="", fp16=True):
    """L2-normalised embeddings as float16 (cosine similarity == dot product)."""
    import torch
    from sentence_transformers import SentenceTransformer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if model_name not in _MODELS:
        _MODELS.clear()
        torch.cuda.empty_cache()
        m = SentenceTransformer(model_name, device=dev)
        if fp16 and dev == "cuda":
            m.half()
        _MODELS[model_name] = m
    m = _MODELS[model_name]
    if max_len:
        m.max_seq_length = max_len
    if prefix:
        texts = [prefix + t for t in texts]
    E = m.encode(list(texts), batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True,
                 show_progress_bar=True)
    return E.astype(np.float16)

Writing encoders.py


In [5]:
%%writefile retrieval.py
"""Phase 2 retrieval helpers: dense search, reciprocal-rank fusion, same-group exclusion, ROUGE-L pair scoring,
chunk-recall metrics. Pure ASCII source on purpose."""
import json
import numpy as np
from multiprocessing import Pool


# ----------------------------------------------------------------------------- dense search
def dense_topk(Q, D, k, batch=512):
    """Q (nq, d) and D (nd, d) are L2-normalised. Returns (idx, score) of the top-k rows of D for every query."""
    k = min(k, D.shape[0])
    nq = Q.shape[0]
    idx = np.zeros((nq, k), dtype=np.int32)
    sc = np.zeros((nq, k), dtype=np.float32)
    try:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        dt = torch.float16 if dev == "cuda" else torch.float32
        Dt = torch.from_numpy(np.ascontiguousarray(D)).to(dev, dtype=dt)
        for i in range(0, nq, batch):
            q = torch.from_numpy(np.ascontiguousarray(Q[i:i + batch])).to(dev, dtype=dt)
            v, ix = (q @ Dt.T).float().topk(k, dim=1)
            idx[i:i + batch] = ix.cpu().numpy()
            sc[i:i + batch] = v.cpu().numpy()
        return idx, sc
    except ImportError:
        pass
    D32 = D.astype(np.float32)
    for i in range(0, nq, batch):
        S = Q[i:i + batch].astype(np.float32) @ D32.T
        part = np.argpartition(-S, k - 1, axis=1)[:, :k]
        ps = np.take_along_axis(S, part, axis=1)
        order = np.argsort(-ps, axis=1)
        idx[i:i + batch] = np.take_along_axis(part, order, axis=1)
        sc[i:i + batch] = np.take_along_axis(ps, order, axis=1)
    return idx, sc


# ----------------------------------------------------------------------------- fusion
def rrf(rank_lists, weights, k=60, topn=50):
    """Weighted reciprocal-rank fusion. rank_lists: list of (nq, depth) int arrays of candidate ids (-1 = none).
    Returns (ids, scores) of shape (nq, topn); ids padded with -1."""
    nq = rank_lists[0].shape[0]
    ids = np.full((nq, topn), -1, dtype=np.int32)
    scs = np.zeros((nq, topn), dtype=np.float32)
    for q in range(nq):
        acc = {}
        for L, w in zip(rank_lists, weights):
            for r, c in enumerate(L[q]):
                if c >= 0:
                    acc[c] = acc.get(c, 0.0) + w / (k + r + 1)
        best = sorted(acc.items(), key=lambda kv: -kv[1])[:topn]
        for j, (c, s) in enumerate(best):
            ids[q, j], scs[q, j] = c, s
    return ids, scs


def drop_groups(ids, scores, pool_group, query_group, topn):
    """Remove candidates whose group equals the query's group (a question must not retrieve itself or its repeats)."""
    out_i = np.full((ids.shape[0], topn), -1, dtype=np.int32)
    out_s = np.zeros((ids.shape[0], topn), dtype=np.float32)
    for q in range(ids.shape[0]):
        j = 0
        for c, s in zip(ids[q], scores[q]):
            if c < 0 or pool_group[c] == query_group[q]:
                continue
            out_i[q, j], out_s[q, j] = c, s
            j += 1
            if j == topn:
                break
    return out_i, out_s


# ----------------------------------------------------------------------------- ROUGE-L on many pairs
_SC = None
CAP_WORDS = 700  # long answers are truncated for selection/tuning (LCS is quadratic); final metrics use full text


def _rl(pair):
    global _SC
    if _SC is None:
        from rouge_score import rouge_scorer
        _SC = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    a, b = pair
    return _SC.score(" ".join(a.split()[:CAP_WORDS]), " ".join(b.split()[:CAP_WORDS]))["rougeL"].fmeasure


def rougeL_pairs(pairs, procs=4):
    if not pairs:
        return []
    with Pool(procs) as p:
        return p.map(_rl, pairs, chunksize=16)


def mbr_pick(answer_lists, procs=4):
    """For each query pick the candidate answer with the highest mean ROUGE-L against the other candidates
    (the 'medoid'). answer_lists: list of lists of strings. Returns the chosen position per query."""
    pairs, owner = [], []
    for qi, al in enumerate(answer_lists):
        for i in range(len(al)):
            for j in range(i + 1, len(al)):
                pairs.append((al[i], al[j]))
                owner.append((qi, i, j))
    scores = rougeL_pairs(pairs, procs)
    tot = [[0.0] * len(al) for al in answer_lists]
    for (qi, i, j), s in zip(owner, scores):
        tot[qi][i] += s
        tot[qi][j] += s
    return [int(np.argmax(t)) if t else 0 for t in tot]


# ----------------------------------------------------------------------------- chunk retrieval metrics
def chunk_gold(citations_json, chunk_pages, chunk_docs):
    """Gold sets for one question: pages (doc, page) and docs cited by the training answer that exist as chunks."""
    pages, docs = set(), set()
    for c in json.loads(citations_json):
        if c.get("kind") != "zoho" or c["doc_id"] not in chunk_docs:
            continue
        docs.add(c["doc_id"])
        for p in c.get("pages", []):
            if (c["doc_id"], p) in chunk_pages:
                pages.add((c["doc_id"], p))
    return pages, docs


def chunk_recall(hits, gold_pages, gold_docs, chunk_doc, chunk_page, ms=(1, 5, 10, 20)):
    """hits: (nq, >=max(ms)) chunk ids. Returns {metric: value} over the questions that have gold of that kind."""
    out = {}
    for m in ms:
        pg_h = pg_n = dc_h = dc_n = 0
        for q in range(hits.shape[0]):
            got = [int(c) for c in hits[q, :m] if c >= 0]
            if gold_pages[q]:
                pg_n += 1
                pg_h += any((chunk_doc[c], chunk_page[c]) in gold_pages[q] for c in got)
            if gold_docs[q]:
                dc_n += 1
                dc_h += any(chunk_doc[c] in gold_docs[q] for c in got)
        out["page_hit@%d" % m] = pg_h / max(1, pg_n)
        out["doc_hit@%d" % m] = dc_h / max(1, dc_n)
        out["n_page_q"], out["n_doc_q"] = pg_n, dc_n
    return out

Writing retrieval.py


## 0. Load data

In [6]:
import glob, os, sys, json, gzip, csv, time, collections as C
import numpy as np
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("eval_harness.py", "bm25.py", "encoders.py", "retrieval.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - run the %%writefile cells at the top first (use Run All, do not start from a lower cell)"
sys.path.insert(0, os.getcwd())

def find(pattern):
    hit = glob.glob("/kaggle/input/**/" + pattern, recursive=True)
    if not hit:  # show what IS attached, so a wrong name / missing dataset is obvious
        print("not found:", pattern, "- files under /kaggle/input (first 40):")
        for i, (dp, dn, fn) in enumerate(os.walk("/kaggle/input")):
            for x in fn[:8]:
                print("  ", os.path.join(dp, x))
            if i > 40:
                break
    assert hit, "attach a dataset containing " + pattern
    return hit[0]

# chunks.jsonl may have been decompressed to plain .jsonl when the dataset was created, so accept both
TRAIN_CSV, UNSEEN_CSV, CHUNKS = find("train_processed*.csv"), find("test_processed*.csv"), find("chunks.jsonl*")
OUT = "/kaggle/working/phase2"
os.makedirs(OUT, exist_ok=True)
QLIMIT = int(os.environ.get("P2_QLIMIT", 0))            # debug hooks only; 0 = use everything
CHUNK_LIMIT = int(os.environ.get("P2_CHUNK_LIMIT", 0))
Q_ENCODER, C_ENCODER = "BAAI/bge-m3", "BAAI/bge-small-en-v1.5"
C_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
DEPTH, TOPN_Q, TOPN_C = 60, 10, 20
print(TRAIN_CSV, "|", UNSEEN_CSV, "|", CHUNKS)

rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
unseen = list(csv.DictReader(open(UNSEEN_CSV, encoding="utf-8")))
split = np.array([r["split"] for r in rows])
grp = np.array([r["group_id"] for r in rows])
norm = [r["norm_q"] for r in rows]
answers = [r["final_answer"] for r in rows]
qids = [r["question_id"] for r in rows]
ix = {s: np.where(split == s)[0] for s in ("train", "dev", "test")}
if QLIMIT:
    ix["dev"], ix["test"], ix["train"] = ix["dev"][:QLIMIT], ix["test"][:QLIMIT], ix["train"][:3 * QLIMIT]
pool_tr = np.where(split == "train")[0]            # the retrieval pool for dev / test / train queries
print({s: len(v) for s, v in ix.items()}, "| pool:", len(pool_tr), "| unseen:", len(unseen))

ch_doc, ch_page, ch_text = [], [], []
for line in (gzip.open(CHUNKS, "rt", encoding="utf-8") if CHUNKS.endswith(".gz") else open(CHUNKS, encoding="utf-8")):
    c = json.loads(line)
    ch_doc.append(c["doc_id"]); ch_page.append(c["page"]); ch_text.append(c["text"])
    if CHUNK_LIMIT and len(ch_text) >= CHUNK_LIMIT:
        break
print("chunks:", len(ch_text))

/kaggle/input/datasets/arth99/train-processed1/train_processed.csv | /kaggle/input/datasets/arth99/train-processed/test_processed.csv | /kaggle/input/datasets/arth99/nlp-qna-chunks/chunks.jsonl
{'train': 12825, 'dev': 675, 'test': 1500} | pool: 12825 | unseen: 1000
chunks: 336122


## 1. Embeddings (GPU)
`bge-m3` for question-to-question matching (multilingual - handles the Gurmukhi/Devanagari questions); `bge-small-en` for question-to-chunk matching (336k chunks - 17x cheaper).

In [7]:
import encoders as enc
t0 = time.time()
all_q = [r["question"] for r in rows] + [r["question"] for r in unseen]
Qm = enc.encode(Q_ENCODER, all_q, batch_size=64)
print("question embeddings (m3):", Qm.shape, "%.1f min" % ((time.time() - t0) / 60))
Qc = enc.encode(C_ENCODER, all_q, prefix=C_QUERY_PREFIX, max_len=64)
Cemb = enc.encode(C_ENCODER, ch_text, batch_size=128, max_len=320)
print("chunk embeddings:", Cemb.shape, "%.1f min total" % ((time.time() - t0) / 60))
np.save(OUT + "/q_emb_m3.npy", Qm); np.save(OUT + "/q_emb_chunkspace.npy", Qc); np.save(OUT + "/chunk_emb.npy", Cemb)
Qm_rows, Qm_unseen = Qm[:len(rows)], Qm[len(rows):]
Qc_rows, Qc_unseen = Qc[:len(rows)], Qc[len(rows):]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (25 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

Batches:   0%|          | 0/2626 [00:00<?, ?it/s]

chunk embeddings: (336122, 384) 12.5 min total


## 2. Index A - training questions: tune the fusion on the dev split
Objective: ROUGE-L between the top-1 neighbour's answer and the true answer (answers truncated to 700 words for speed).

In [8]:
from bm25 import BM25
import retrieval as R
bm_tr = BM25().fit([norm[i] for i in pool_tr])
D_tr = Qm_rows[pool_tr]
dev = ix["dev"]
b_i, _ = bm_tr.search([norm[i] for i in dev], k=DEPTH)
d_i, _ = R.dense_topk(Qm_rows[dev], D_tr, k=DEPTH)
gold_dev = [answers[i] for i in dev]

def top1_rougeL(ids):
    pairs = [(answers[pool_tr[c]] if c >= 0 else "", gold_dev[q]) for q, c in enumerate(ids[:, 0])]
    return float(np.mean(R.rougeL_pairs(pairs)))

rng = np.random.RandomState(0)
tune = {"random train answer (floor)": float(np.mean(R.rougeL_pairs(
            [(answers[pool_tr[rng.randint(len(pool_tr))]], g) for g in gold_dev]))),
        "BM25 only": top1_rougeL(b_i), "dense (bge-m3) only": top1_rougeL(d_i)}
fused = {}
for rk in (20, 60):
    for wd in (1.0, 2.0):
        ids, _ = R.rrf([b_i, d_i], [1.0, wd], k=rk, topn=10)
        name = "RRF k=%d  w_dense=%.0f" % (rk, wd)
        tune[name] = top1_rougeL(ids); fused[name] = (rk, wd, ids)
best_name = max(fused, key=lambda n: tune[n])
BEST_K, BEST_W, best_ids = fused[best_name]
print("%-30s %s" % ("retriever (dev, %d questions)" % len(dev), "top-1 ROUGE-L"))
for n, v in tune.items():
    print("%-30s %.4f%s" % (n, v, "   <- chosen" if n == best_name else ""))
# how much headroom is there if we could pick the best of the top-5 neighbours?
pairs = [(answers[pool_tr[c]], gold_dev[q]) for q in range(len(dev)) for c in best_ids[q, :5] if c >= 0]
sc5 = R.rougeL_pairs(pairs); i = 0; orc = []
for q in range(len(dev)):
    m = sum(1 for c in best_ids[q, :5] if c >= 0); orc.append(max(sc5[i:i + m]) if m else 0.0); i += m
print("oracle best-of-5 neighbours: %.4f" % np.mean(orc))

retriever (dev, 675 questions) top-1 ROUGE-L
random train answer (floor)    0.1141
BM25 only                      0.1965
dense (bge-m3) only            0.2149
RRF k=20  w_dense=1            0.2058
RRF k=20  w_dense=2            0.2114   <- chosen
RRF k=60  w_dense=1            0.2047
RRF k=60  w_dense=2            0.2105
oracle best-of-5 neighbours: 0.2729


## 3. Index A - neighbours for every query set (saved for Phase 4)
train queries exclude their own group; dev/test use the train pool; the unseen set uses all 15,000 questions.

In [9]:
bm_all = BM25().fit(norm)

def neighbours(q_norm, q_dense, bm, D, pool_rows, q_group=None):
    b, _ = bm.search(q_norm, k=DEPTH)
    d, _ = R.dense_topk(q_dense, D, k=DEPTH)
    ids, sc = R.rrf([b, d], [1.0, BEST_W], k=BEST_K, topn=DEPTH)
    if q_group is not None:
        return R.drop_groups(ids, sc, grp[pool_rows], q_group, TOPN_Q)
    return ids[:, :TOPN_Q], sc[:, :TOPN_Q]

NN = {}
for s in ("train", "dev", "test"):
    q = ix[s]
    NN[s] = neighbours([norm[i] for i in q], Qm_rows[q], bm_tr, D_tr, pool_tr, grp[q] if s == "train" else None)
NN["xt"] = neighbours([r["norm_q"] for r in unseen], Qm_unseen, bm_all, Qm_rows, np.arange(len(rows)))
pool_of = {"train": pool_tr, "dev": pool_tr, "test": pool_tr, "xt": np.arange(len(rows))}
q_ids_of = {s: [qids[i] for i in ix[s]] for s in ("train", "dev", "test")}
q_ids_of["xt"] = [r["question_id"] for r in unseen]
out = {s: {q_ids_of[s][k]: [[qids[pool_of[s][c]], round(float(sc), 5)] for c, sc in zip(NN[s][0][k], NN[s][1][k]) if c >= 0]
           for k in range(len(q_ids_of[s]))} for s in NN}
with gzip.open(OUT + "/question_neighbours.json.gz", "wt", encoding="utf-8") as f:
    json.dump(out, f)
print({s: len(v) for s, v in out.items()}, "| example:", list(out["dev"].items())[0])

{'train': 12825, 'dev': 675, 'test': 1500, 'xt': 1000} | example: ('6a2fcfc2604713841b6b97b1', [['6a435a2ed5bbcaa6bc820301', 0.13095], ['6a2fd7de604713841b6b9c9d', 0.11485], ['6a3fa111b7ec8be390de91f4', 0.10897], ['6a3e080bcafbea11942169c3', 0.10173], ['6a3fb5a8b7ec8be390de956c', 0.09792], ['69ab9d5ec2b602929b370726', 0.09107], ['6a2fcf4e604713841b6b9789', 0.09091], ['6a2fcec9604713841b6b9751', 0.08777], ['6a300d0682b99d966ae083e4', 0.08696], ['6a33d0b5dd4aa34618b54400', 0.08348]])


## 4. Index B - PDF chunks: BM25 + dense, scored against the pages the training answers cite

In [10]:
t0 = time.time()
bm_c = BM25().fit(ch_text)
print("chunk BM25 fitted: %.1f min" % ((time.time() - t0) / 60))
q_texts = [r["norm_q"] for r in rows] + [r["norm_q"] for r in unseen]
cb_i, _ = bm_c.search(q_texts, k=DEPTH)
cd_i, _ = R.dense_topk(np.concatenate([Qc_rows, Qc_unseen]), Cemb, k=DEPTH)
print("chunk retrieval for all %d queries: %.1f min" % (len(q_texts), (time.time() - t0) / 60))

chunk_pages, chunk_docs = set(zip(ch_doc, ch_page)), set(ch_doc)
ev = np.concatenate([ix["dev"], ix["test"]])
gp, gd = zip(*[R.chunk_gold(rows[i]["citations_json"], chunk_pages, chunk_docs) for i in ev])
variants = {"BM25 only": cb_i[ev], "dense (bge-small) only": cd_i[ev]}
cfused = {}
for wd in (1.0, 2.0):
    ids, _ = R.rrf([cb_i[ev], cd_i[ev]], [1.0, wd], k=60, topn=TOPN_C)
    variants["RRF w_dense=%.0f" % wd] = ids; cfused[wd] = ids
recall = {n: R.chunk_recall(h, gp, gd, ch_doc, ch_page) for n, h in variants.items()}
print("questions with a gold page: %d | with a gold doc: %d (of %d dev+test)" % (
    recall["BM25 only"]["n_page_q"], recall["BM25 only"]["n_doc_q"], len(ev)))
print("%-24s %s" % ("", "  ".join("page@%-2d doc@%-2d" % (m, m) for m in (1, 5, 10, 20))))
for n, r in recall.items():
    print("%-24s %s" % (n, "  ".join("%.2f   %.2f " % (r["page_hit@%d" % m], r["doc_hit@%d" % m]) for m in (1, 5, 10, 20))))
BEST_CW = max(cfused, key=lambda w: recall["RRF w_dense=%.0f" % w]["page_hit@10"])
print("chosen chunk fusion: RRF w_dense=%.0f" % BEST_CW)

chunk BM25 fitted: 0.8 min
chunk retrieval for all 16000 queries: 2.4 min
questions with a gold page: 944 | with a gold doc: 1749 (of 2175 dev+test)
                         page@1  doc@1   page@5  doc@5   page@10 doc@10  page@20 doc@20
BM25 only                0.02   0.03   0.05   0.08   0.08   0.13   0.11   0.18 
dense (bge-small) only   0.04   0.06   0.08   0.14   0.11   0.19   0.14   0.26 
RRF w_dense=1            0.04   0.05   0.07   0.12   0.10   0.18   0.15   0.27 
RRF w_dense=2            0.04   0.05   0.09   0.14   0.11   0.20   0.14   0.27 
chosen chunk fusion: RRF w_dense=2


In [11]:
cids, _ = R.rrf([cb_i, cd_i], [1.0, BEST_CW], k=60, topn=TOPN_C)
hits = {"train": {}, "dev": {}, "test": {}, "xt": {}}
for s in ("train", "dev", "test"):
    for i in ix[s]:
        hits[s][qids[i]] = [int(c) for c in cids[i] if c >= 0]
for k, r in enumerate(unseen):
    hits["xt"][r["question_id"]] = [int(c) for c in cids[len(rows) + k] if c >= 0]
with gzip.open(OUT + "/chunk_hits.json.gz", "wt", encoding="utf-8") as f:
    json.dump(hits, f)   # chunk ids = line numbers (0-based) of chunks.jsonl.gz
print({s: len(v) for s, v in hits.items()})

{'train': 12825, 'dev': 675, 'test': 1500, 'xt': 1000}


## 5. B0 baselines on the 1,500 hold-out rows
B0a: answer = the nearest training question's answer. B0b: the medoid (highest mean ROUGE-L vs the others) of the 5 nearest answers.

In [12]:
from eval_harness import evaluate
import csv as _csv
te = ix["test"]
refs = {qids[i]: answers[i] for i in te}
ids_te = NN["test"][0]
b0a = {qids[i]: answers[pool_tr[ids_te[k, 0]]] if ids_te[k, 0] >= 0 else "" for k, i in enumerate(te)}
cand = [[answers[pool_tr[c]] for c in ids_te[k, :5] if c >= 0] or [""] for k in range(len(te))]
pick = R.mbr_pick(cand)
b0b = {qids[i]: cand[k][pick[k]] for k, i in enumerate(te)}
res = {"B0a nearest-neighbour answer": evaluate(b0a, refs), "B0b medoid of top-5": evaluate(b0b, refs)}
cols = ["rouge1", "rouge2", "rougeL", "bleu", "len_ratio_mean", "fact_recall", "fact_precision", "fact_f1"]
print("%-32s %s" % ("hold-out test (n=%d)" % len(te), "  ".join("%-9s" % c[:9] for c in cols)))
for n, m in res.items():
    print("%-32s %s" % (n, "  ".join("%-9.4f" % m[c] for c in cols)))
with open(OUT + "/b0_test_predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = _csv.writer(f); w.writerow(["question_id", "b0a_nearest", "b0b_medoid5"])
    for q in refs:
        w.writerow([q, b0a[q], b0b[q]])

hold-out test (n=1500)           rouge1     rouge2     rougeL     bleu       len_ratio  fact_reca  fact_prec  fact_f1  
B0a nearest-neighbour answer     0.4088     0.1496     0.2171     0.1407     1.0140     0.1911     0.1863     0.1684   
B0b medoid of top-5              0.4069     0.1447     0.2133     0.1326     0.9615     0.1792     0.1856     0.1607   


## 6. Save the report

In [13]:
report = {"encoders": {"questions": Q_ENCODER, "chunks": C_ENCODER}, "rrf": {"k": BEST_K, "w_dense": BEST_W, "chunk_w_dense": BEST_CW},
          "dev_tuning_top1_rougeL": tune, "oracle_best_of_5": float(np.mean(orc)),
          "chunk_retrieval_recall": recall, "b0_holdout": res,
          "sizes": {"train_pool": len(pool_tr), "chunks": len(ch_text), "unseen": len(unseen)}}
json.dump(report, open(OUT + "/phase2_report.json", "w"), indent=1)
print(sorted(os.listdir(OUT)))
print("total MB:", sum(os.path.getsize(OUT + "/" + f) for f in os.listdir(OUT)) // 2 ** 20)

['b0_test_predictions.csv', 'chunk_emb.npy', 'chunk_hits.json.gz', 'phase2_report.json', 'q_emb_chunkspace.npy', 'q_emb_m3.npy', 'question_neighbours.json.gz']
total MB: 301


## After it finishes
Output tab -> **New Dataset** (e.g. `nlp-qna-retrieval`). Phase 4 (training) attaches it for the neighbour lists, chunk hits and embeddings.
Keep the printed tables - the dev tuning table, the chunk-recall table and the B0 table go straight into the report.